# Tool-Calling Research Agent (Teacher)

**Phase 1** of the BBB conference demo pipeline.

This notebook implements a simple tool-calling agent using the OpenAI API.
The agent researches a given stock ticker by calling financial tools (yfinance),
then produces a structured research memo.

The full conversation trajectory (including all tool calls and results) is saved
as training data for fine-tuning a smaller open-source model (Qwen3-4B).

## Architecture
```
User prompt → GPT (teacher) → tool_call → execute tool → result → GPT → ... → final memo
                                ↑                                        |
                                └── full trajectory saved as JSONL ──────┘
```

## Setup

In [1]:
import json
import os
import sys
from pathlib import Path

from openai import OpenAI
from dotenv import load_dotenv

# Add project root to path so we can import tools
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from tools.stock_tools import (
    TOOL_SCHEMAS,
    TOOL_FUNCTIONS,
    run_tool_calling_agent,
)

load_dotenv(PROJECT_ROOT / ".env")

client = OpenAI(base_url="https://us.api.openai.com/v1")

# Teacher model — change to gpt-5.4 when available
MODEL = "gpt-5.4"

## System Prompt

Instructs the model to:
1. Act as an equity research analyst
2. Use tools to gather data
3. Think before each tool call (in `<think>` tags)
4. Produce a structured research memo as final output

In [14]:
SYSTEM_PROMPT = """\
You are an equity research analyst conducting comprehensive research on a given stock.

## Instructions
- Use the available tools to gather financial data, news, price history, and analyst recommendations.
- Before each tool call, take time to think about the task at hand.
- After gathering sufficient data, synthesize your findings into a structured research memo.
- Be thorough but efficient — aim for just the necessary amount of tool calls.

## Output Format
Your final response should be a markdown research memo with these sections:
- **Company Overview**: Brief description and market position
- **Financial Analysis**: Revenue, profitability, balance sheet highlights
- **Market Performance**: Price trends, volatility, trading patterns
- **Analyst Sentiment**: Consensus recommendations, recent upgrades/downgrades
- **Key Risks & Opportunities**: Summary of bull/bear case

## Important
- Use ONLY the tools provided. Do not make up financial data.
- Present facts objectively. Do not give investment advice.
- If a tool returns an error, note it and move on.
"""

## Tool Schemas

These are defined in `tools/stock_tools.py` and shared across all BBB notebooks.

In [15]:
# Quick look at what tools the agent has access to
for tool in TOOL_SCHEMAS:
    fn = tool["function"]
    params = list(fn["parameters"]["properties"].keys())
    print(f"  {fn['name']}({', '.join(params)})")
    print(f"    → {fn['description']}\n")

  get_stock_news(ticker)
    → Get the 5 most recent news articles for a stock ticker from Yahoo Finance.

  get_financials(ticker, statement_type, period)
    → Get financial statements (income statement, balance sheet, or cash flow) for a stock.

  get_price_history(ticker, period, interval)
    → Get historical stock price data (OHLCV) with summary statistics.

  get_recommendations(ticker, months_back)
    → Get analyst recommendations and recent upgrades/downgrades for a stock.



## Run the Agent

Let's test with a single ticker first.

In [16]:
trajectory = run_tool_calling_agent(
    client=client,
    model=MODEL,
    user_prompt="Research NVDA focusing on financial health and growth potential.",
    system_prompt=SYSTEM_PROMPT,
)

print(f"\nTrajectory has {len(trajectory)} messages")
print(f"Tool calls: {sum(1 for m in trajectory if m.get('role') == 'tool')}")

  [1] Called get_financials(ticker='NVDA', statement_type='income', period='annual')
  [1] Called get_financials(ticker='NVDA', statement_type='balance_sheet', period='annual')
  [1] Called get_financials(ticker='NVDA', statement_type='cashflow', period='annual')
  [1] Called get_price_history(ticker='NVDA', period='1y', interval='1wk')
  [1] Called get_recommendations(ticker='NVDA', months_back=12)
  [1] Called get_stock_news(ticker='NVDA')
  [2] Agent finished — produced final response

Trajectory has 10 messages
Tool calls: 6


### Inspect the trajectory

In [20]:
# Print each message role and a preview
for i, msg in enumerate(trajectory):
    role = msg["role"]
    content = msg.get("content", "")
    tool_calls = msg.get("tool_calls", [])

    if role == "system":
        print(f"[{i}] SYSTEM: {content[:80]}...\n")
    elif role == "user":
        print(f"[{i}] USER: {content}\n")
    elif role == "assistant":
        think = content[:100] if content else "(no content)"
        print(f"[{i}] ASSISTANT [think]: {think}\n")
        if tool_calls:
            names = [tc["function"]["name"] for tc in tool_calls]
            print(f"    [calls]: {names}\n")
            #print(f"    [full message]: {content[:120]}...\n")
        else:
            print(f"    (final message): {content[:120]}...\n")
    elif role == "tool":
        print(f"[{i}] TOOL ({msg.get('tool_call_id', '?')[:12]}): {content[:80]}...\n")

[0] SYSTEM: You are an equity research analyst conducting comprehensive research on a given ...

[1] USER: Research NVDA focusing on financial health and growth potential.

[2] ASSISTANT [think]: I’ll gather NVDA’s latest financial statements, price history, analyst recommendations, and recent n

    [calls]: ['get_financials', 'get_financials', 'get_financials', 'get_price_history', 'get_recommendations', 'get_stock_news']

[3] TOOL (call_Yoo9FnO): {
  "ticker": "NVDA",
  "statement_type": "income",
  "period": "annual",
  "dat...

[4] TOOL (call_G5Y0gXI): {
  "ticker": "NVDA",
  "statement_type": "balance_sheet",
  "period": "annual",...

[5] TOOL (call_0FW28hD): {
  "ticker": "NVDA",
  "statement_type": "cashflow",
  "period": "annual",
  "d...

[6] TOOL (call_lEAwYYp): {
  "ticker": "NVDA",
  "period": "1y",
  "interval": "1wk",
  "data_points": 53...

[7] TOOL (call_yv8BBUt): {
  "ticker": "NVDA",
  "recommendations": [
    {
      "index": 0,
      "peri...

[8] TOOL (call_tV0HCa

### View the final memo

In [18]:
from IPython.display import Markdown, display

# The last assistant message (without tool_calls) is the final memo
final_msg = [m for m in trajectory if m["role"] == "assistant" and not m.get("tool_calls")][-1]
display(Markdown(final_msg["content"]))

# Research Memo: NVIDIA (NASDAQ: NVDA)

## Company Overview
NVIDIA is a leading semiconductor and computing platform company best known for GPUs, AI accelerators, networking, and software platforms used across data centers, gaming, autonomous systems, and enterprise AI infrastructure.

Its market position is currently defined by strong leadership in AI compute, especially in data center training and inference workloads. Financially, the company appears to be operating from a position of unusual strength: very high revenue growth, expanding profitability, large free cash flow generation, and a balance sheet with substantial liquidity.

## Financial Analysis

### Revenue and growth
NVIDIA’s annual revenue has risen sharply over the last four reported fiscal years:

- **FY2023 (ended Jan. 2023):** **$26.97B**
- **FY2024:** **$60.92B**
- **FY2025:** **$130.50B**
- **FY2026:** **$215.94B**

That implies:
- **FY2024 YoY growth:** about **126%**
- **FY2025 YoY growth:** about **114%**
- **FY2026 YoY growth:** about **65%**

Growth remains extremely strong, though naturally decelerating from the extraordinary AI ramp of the prior two years. Even with that slowdown, a 65% revenue increase on a very large base is notable.

### Profitability
Profitability has expanded dramatically:

- **Gross profit FY2026:** **$153.46B**
- **Operating income FY2026:** **$130.39B**
- **Net income FY2026:** **$120.07B**
- **Diluted EPS FY2026:** **$4.90**

Margin progression is also striking:

- **Gross margin FY2026:** about **71.1%**
- **Operating margin FY2026:** about **60.4%**
- **Net margin FY2026:** about **55.6%**

For comparison:
- FY2025 net income: **$72.88B**
- FY2024 net income: **$29.76B**
- FY2023 net income: **$4.37B**

This indicates not just top-line growth, but exceptional operating leverage. R&D also continues to scale:
- **R&D FY2026:** **$18.50B**
- vs. **$12.91B** in FY2025
- vs. **$8.68B** in FY2024

That suggests NVIDIA is still investing heavily while expanding margins.

### Cash flow and capital allocation
Cash generation is very strong:

- **Operating cash flow FY2026:** **$102.72B**
- **Free cash flow FY2026:** **$96.68B**

Historical FCF trend:
- **FY2024:** **$27.02B**
- **FY2025:** **$60.85B**
- **FY2026:** **$96.68B**

Capital spending is rising but still modest relative to operating cash flow:
- **Capex FY2026:** **$6.04B**

Capital returns are substantial:
- **Share repurchases FY2026:** **$40.09B**
- **Dividends paid FY2026:** **$974M**

One point to watch: working capital consumed cash in FY2026, driven by:
- **Receivables increase:** about **$15.40B**
- **Inventory increase:** about **$11.32B**

This is not necessarily negative in a fast-growing hardware cycle, but it does show that growth is increasingly tied to larger balance-sheet commitments.

### Balance sheet and financial health
NVIDIA’s balance sheet appears very healthy.

#### Liquidity
- **Cash, cash equivalents, and short-term investments FY2026:** **$62.56B**
- **Current assets:** **$125.61B**
- **Current liabilities:** **$32.16B**
- **Working capital:** **$93.44B**

Approximate current ratio:
- **125.61 / 32.16 ≈ 3.9x**

That indicates strong short-term liquidity.

#### Debt and equity
- **Total debt FY2026:** **$11.04B**
- **Stockholders’ equity FY2026:** **$157.29B**

Debt is small relative to cash resources and equity. NVIDIA also holds:
- **Available-for-sale securities/investments:** **$22.25B**

The company therefore appears to have a **net cash-like financial profile**, even though a direct net debt line was not populated in the latest dataset.

#### Other balance-sheet observations
- **Accounts receivable FY2026:** **$38.47B** vs. **$23.07B** in FY2025
- **Inventory FY2026:** **$21.40B** vs. **$10.08B** in FY2025
- **Goodwill FY2026:** **$20.83B** and total intangibles/goodwill **$24.14B**

The jump in goodwill/intangibles suggests acquisition activity; cash flow data shows:
- **Business purchases FY2026:** **$14.54B**

Overall, financial health is strong, but investors should monitor whether receivables and inventory growth remain aligned with end-demand.

## Market Performance

### Price trends
Over the last 1 year, NVDA’s weekly price summary shows:

- **Min price:** **$94.29**
- **Max price:** **$202.47**
- **Average price:** **$165.76**
- **Total change over period:** **+56.18%**

The stock experienced:
1. A sharp drawdown early in the period, bottoming near **$94**
2. A strong recovery through mid- to late-2025
3. A rally to a high near **$202**
4. A more recent pullback, ending near **$171.24**

### Volatility and trading patterns
Trading has been volatile, with several weeks above or near **1B shares** in volume, especially during large moves. Examples:
- Week of **2025-04-07**: volume about **2.45B**
- Week of **2025-03-31**: volume about **1.61B**
- Week of **2025-11-17**: volume about **1.32B**
- Week of **2026-02-23**: volume about **1.27B**

This pattern suggests:
- high institutional participation,
- sensitivity to earnings/AI demand expectations,
- and elevated sentiment-driven swings despite strong fundamentals.

The recent pullback from the peak to the latest close implies the market may be reassessing valuation, durability of AI demand, or broader tech risk appetite.

## Analyst Sentiment

### Consensus snapshot
Current recommendation mix for **0 months**:
- **Strong Buy:** 9
- **Buy:** 48
- **Hold:** 2
- **Sell:** 1
- **Strong Sell:** 0

This is overwhelmingly positive. Across the last few months, the recommendation profile has remained similarly strong.

### Recent rating actions
Recent actions are mostly reaffirmations and price-target increases:

- **Raymond James:** Strong Buy, PT raised to **$323**
- **Truist:** Buy, PT raised to **$287**
- **Tigress Financial:** Strong Buy, PT raised to **$360**
- **Wedbush:** Outperform, PT raised to **$300**
- **JP Morgan:** Overweight, PT raised to **$265**
- **Citigroup:** Buy, PT raised to **$300**
- **Morgan Stanley:** Overweight, PT raised to **$260**
- **Bernstein:** Outperform, PT raised to **$300**
- **BofA Securities:** Buy, PT raised to **$300**

There are some more cautious views in the history:
- **Deutsche Bank:** Hold, PT raised to **$215**
- **Seaport Global:** initiated **Sell** with PT **$100** in April 2025

Net takeaway: analyst sentiment remains highly constructive, with the dominant view that NVIDIA retains strong earnings power and AI-related upside.

## Key Risks & Opportunities

### Opportunities / Bull case
- **Sustained AI infrastructure demand:** Revenue growth from **$60.9B to $215.9B** in two years shows NVIDIA remains the central beneficiary of AI capex.
- **Exceptional profitability:** FY2026 net margin above **55%** and operating margin above **60%** suggest strong pricing power and product mix.
- **Large free cash flow engine:** Nearly **$96.7B** in FCF provides flexibility for R&D, acquisitions, and buybacks.
- **Strong balance sheet:** Over **$62.6B** in cash and short-term investments against **$11.0B** in total debt supports resilience.
- **Continued investment in innovation:** **$18.5B** in R&D suggests NVIDIA is trying to preserve its lead in AI hardware and software ecosystems.

### Risks / Bear case
- **Growth deceleration risk:** Revenue growth slowed from triple digits to about **65%** in FY2026; further normalization could pressure valuation.
- **Working capital build:** Large increases in receivables and inventory may indicate execution risk if customer demand or deployment timing weakens.
- **Valuation sensitivity:** The stock’s wide trading range and sharp pullbacks show that even strong fundamentals may not prevent multiple compression.
- **Customer concentration / AI cycle risk:** Results are likely heavily tied to hyperscaler and enterprise AI spending trends.
- **Competitive and platform risk:** Any meaningful gains by rival chipmakers or changes in AI compute architectures could affect long-term margins.
- **Acquisition integration risk:** FY2026 included **$14.5B** of business purchases, which can create execution and balance-sheet complexity.

## Bottom Line
NVIDIA’s financial health appears exceptionally strong: rapid revenue expansion, elite margins, enormous free cash flow, and a highly liquid balance sheet. Growth potential remains compelling given its central role in AI infrastructure, though the key debate is shifting from **whether growth is strong** to **how durable and scalable that growth remains from a much larger base**.

The main tension in the story is clear: fundamentals are outstanding, but expectations are also very high.

## Save the Trajectory

Save in the format we'll use for training data.

In [ ]:
output_dir = PROJECT_ROOT / "data" / "bbb"
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "tool_calling_trajectories.jsonl"

record = {
    "messages": trajectory,
    "tools": TOOL_SCHEMAS,
}

with open(output_file, "a") as f:
    f.write(json.dumps(record) + "\n")

print(f"Saved trajectory to {output_file}")

## Test with a Few More Tickers

In [ ]:
test_tickers = [
    ("AAPL", "competitive position and market share"),
    ("TSLA", "recent news and analyst sentiment"),
    ("JPM", "financial health and balance sheet strength"),
]

for ticker, focus in test_tickers:
    print(f"\n{'='*60}")
    print(f"Researching {ticker} — {focus}")
    print(f"{'='*60}")
    
    traj = run_tool_calling_agent(
        client=client,
        model=MODEL,
        user_prompt=f"Research {ticker} focusing on {focus}.",
        system_prompt=SYSTEM_PROMPT,
    )
    
    n_tools = sum(1 for m in traj if m.get("role") == "tool")
    final = [m for m in traj if m["role"] == "assistant" and not m.get("tool_calls")]
    memo_len = len(final[-1]["content"]) if final else 0
    print(f"  → {n_tools} tool calls, memo length: {memo_len} chars")
    
    # Save
    record = {"messages": traj, "tools": TOOL_SCHEMAS}
    with open(output_file, "a") as f:
        f.write(json.dumps(record) + "\n")